# PJM West Hourly Energy Forecasting

**P679 | EDA, four-model comparison, 30-day forecast, and deployment**

Run this notebook from top to bottom. It loads the supplied workbook, explores hourly and seasonal demand (including a holiday proxy), compares Linear Regression, Random Forest, XGBoost, and SARIMA on the last 365 days, forecasts 30 days, and saves the fitted models for the Streamlit app.

**Date limitation:** the workbook ends on 2018-08-03. The generated forecast is a historical demonstration, not a current operational prediction.

### Requirement steps
1. Data setup and validation
2. EDA: hour, weekday, month, long-term, summer/winter, holidays
3. Four models with a chronological holdout
4. Next 30 days of hourly forecasts
5. Generate model bundle for local/cloud deployment


## 0. Setup

In a terminal opened at the project folder, run: python -m pip install -r requirements.txt. Then open this notebook in VS Code or Jupyter and choose that same Python environment as the kernel. Restart the kernel after installation if needed.


In [ ]:
# Optional, from inside Jupyter if dependencies are not installed:
# %pip install -r requirements.txt

import warnings
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from pandas.tseries.holiday import USFederalHolidayCalendar
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore", category=UserWarning)
plt.style.use("seaborn-v0_8-whitegrid")
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "PJMW_MW_Hourly.xlsx").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Place this notebook beside PJMW_MW_Hourly.xlsx or start Jupyter in the project folder.")
DATA_PATH = ROOT / "PJMW_MW_Hourly.xlsx"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
DEPLOY_DIR = ROOT / "deployment"
print("Project folder:", ROOT)


## 1. Load and validate the hourly series

Duplicate timestamps are averaged. Missing hours are made explicit in the hourly index. Observed points are used for fitting and scoring; SARIMA's training window is interpolated to make it a continuous hourly series.


In [ ]:
raw = pd.read_excel(DATA_PATH, usecols=["Datetime", "PJMW_MW"], parse_dates=["Datetime"])
raw = raw.dropna(subset=["Datetime", "PJMW_MW"]).sort_values("Datetime")
duplicate_count = int(raw["Datetime"].duplicated().sum())
hourly = raw.groupby("Datetime")["PJMW_MW"].mean().sort_index().asfreq("h")
observed = hourly.dropna()
display(pd.Series({
    "Source rows": len(raw),
    "Duplicate timestamps averaged": duplicate_count,
    "Timeline hours": len(hourly),
    "Missing hourly timestamps": int(hourly.isna().sum()),
    "Observed hours": len(observed),
    "Start": observed.index.min(),
    "End": observed.index.max(),
    "Minimum MW": observed.min(),
    "Mean MW": observed.mean(),
    "Maximum MW": observed.max(),
}, name="Value").to_frame())


## 2. Exploratory data analysis

The federal holiday flag is a broad US proxy. PJM covers multiple states, and the source does not include local holiday or operating calendars.


In [ ]:
profile = observed.rename("PJMW_MW").to_frame()
profile["hour"] = profile.index.hour
profile["weekday"] = profile.index.day_name()
profile["month"] = profile.index.month
holiday_dates = USFederalHolidayCalendar().holidays(
    start=observed.index.min().normalize(), end=observed.index.max().normalize()).normalize()
profile["is_federal_holiday"] = profile.index.normalize().isin(holiday_dates)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
profile.groupby("hour")["PJMW_MW"].mean().plot(ax=axes[0, 0], title="Average load by hour")
axes[0, 0].set(xlabel="Hour", ylabel="MW")
weekdays = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
profile.groupby("weekday")["PJMW_MW"].mean().reindex(weekdays).plot(
    kind="bar", ax=axes[0, 1], title="Average load by weekday")
axes[0, 1].set(xlabel="", ylabel="MW")
profile.groupby("month")["PJMW_MW"].mean().plot(kind="bar", ax=axes[1, 0], title="Average load by month")
axes[1, 0].set(xlabel="Month", ylabel="MW")
observed.resample("MS").mean().plot(ax=axes[1, 1], title="Long-term monthly average")
axes[1, 1].set(xlabel="Year", ylabel="MW")
fig.tight_layout()
plt.show()

# Daily shapes across months show how summer and winter differ.
month_hour = profile.pivot_table(index="hour", columns="month", values="PJMW_MW", aggfunc="mean")
fig, ax = plt.subplots(figsize=(13, 5))
im = ax.imshow(month_hour.to_numpy(), aspect="auto", origin="lower", cmap="viridis")
ax.set(title="Average hourly load by month", xlabel="Month", ylabel="Hour of day",
       xticks=range(12), xticklabels=["Jan", "Feb", "Mar", "Apr", "May", "Jun",
       "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
fig.colorbar(im, ax=ax, label="MW")
plt.show()
display(profile.groupby("is_federal_holiday")["PJMW_MW"].agg(["count", "mean", "median"]).rename(
    index={False: "Non-holiday", True: "US federal holiday"}))
display(profile.groupby("month")["PJMW_MW"].agg(["mean", "median", "min", "max"]).round(1))


## 3. Features and chronological holdout

The last 365 days are held out. No random split is used. Regression and tree models share the same calendar features, including hour/day cycles, annual Fourier seasonality, trend, weekend/evening indicators, and federal holiday indicator.


In [ ]:
def make_features(index, origin):
    index = pd.DatetimeIndex(index)
    hour, dow, doy = index.hour.to_numpy(), index.dayofweek.to_numpy(), index.dayofyear.to_numpy()
    elapsed = (index - origin).total_seconds().to_numpy() / (365.2425 * 86400)
    cols = [np.ones(len(index)), elapsed]
    for period, harmonics, values in ((24, 3, hour), (7, 2, dow), (365.2425, 3, doy - 1)):
        for k in range(1, harmonics + 1):
            angle = 2 * np.pi * k * values / period
            cols.extend([np.sin(angle), np.cos(angle)])
    cols.extend([(dow >= 5).astype(float), ((hour >= 16) & (hour <= 20)).astype(float)])
    holidays = USFederalHolidayCalendar().holidays(
        start=index.min().normalize(), end=index.max().normalize()).normalize()
    cols.append(index.normalize().isin(holidays).astype(float))
    return np.column_stack(cols)

origin = observed.index.min()
cutoff = observed.index.max() - pd.Timedelta(days=365)
train, test = observed.loc[observed.index < cutoff], observed.loc[observed.index >= cutoff]
X_train, X_test = make_features(train.index, origin), make_features(test.index, origin)
y_train, y_test = train.to_numpy(), test.to_numpy()
print(f"Train: {train.index.min()} to {train.index.max()} ({len(train):,} observed hours)")
print(f"Test:  {test.index.min()} to {test.index.max()} ({len(test):,} observed hours)")


## 4. Train Linear Regression, Random Forest, and XGBoost

All three use the same training rows, features, and holdout timestamps. Fixed random seeds make the tree models reproducible.


In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=160, min_samples_leaf=2,
        max_features=0.8, n_jobs=-1, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=350, max_depth=7, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.85, objective="reg:squarederror",
        n_jobs=-1, random_state=42, verbosity=0),
}
test_predictions = {}
for name, model in models.items():
    print("Training", name)
    model.fit(X_train, y_train)
    test_predictions[name] = model.predict(X_test)


## 5. Train SARIMA

SARIMA(1,0,1)(1,0,1,24) captures hourly dependence with a 24-hour seasonal cycle. For fitting speed, it uses the latest two years of the training period. The year-long test forecast is issued from the cutoff and is not updated using test observations.


In [ ]:
sarima_start = train.index.max() - pd.Timedelta(days=730)
sarima_train = hourly.loc[(hourly.index < cutoff) & (hourly.index >= sarima_start)].interpolate(method="time")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    sarima_model = SARIMAX(sarima_train, order=(1, 0, 1), seasonal_order=(1, 0, 1, 24),
        trend="c", enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, maxiter=60)
sarima_index = pd.date_range(cutoff, test.index.max(), freq="h")
sarima_test = sarima_model.get_forecast(steps=len(sarima_index)).predicted_mean
test_predictions["SARIMA"] = pd.Series(np.asarray(sarima_test), index=sarima_index).reindex(test.index).to_numpy()
models["SARIMA"] = sarima_model
print("SARIMA training observations:", len(sarima_train))


## 6. Compare models on the holdout

MAE and RMSE are measured in MW; MAPE is a percentage. Weekly seasonal naive is included as a reference baseline, not as one of the four requested fitted models.


In [ ]:
def metric_row(name, actual, predicted):
    error = np.asarray(predicted) - np.asarray(actual)
    return {"Model": name, "N": len(actual), "MAE_MW": np.mean(np.abs(error)),
        "RMSE_MW": np.sqrt(np.mean(error ** 2)),
        "MAPE_pct": np.mean(np.abs(error) / np.maximum(np.abs(actual), 1)) * 100}

comparison = pd.DataFrame([metric_row(name, y_test, pred) for name, pred in test_predictions.items()])
weekly = hourly.reindex(test.index - pd.Timedelta(days=7)).to_numpy()
valid = np.isfinite(weekly)
comparison = pd.concat([comparison, pd.DataFrame([metric_row("Weekly seasonal naive",
    y_test[valid], weekly[valid])])], ignore_index=True).sort_values("MAE_MW")
display(comparison.round(2))
comparison.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)


## 7. Forecast the next 30 days

The 720 hourly forecasts start one hour after the last timestamp in the workbook. Since the source ends in 2018, these forecast dates are historical.


In [ ]:
future_index = pd.date_range(hourly.index.max() + pd.Timedelta(hours=1), periods=30 * 24, freq="h")
future_X = make_features(future_index, origin)
forecast = pd.DataFrame({"Datetime": future_index})
for name in ["Linear Regression", "Random Forest", "XGBoost"]:
    forecast[name.replace(" ", "_") + "_MW"] = models[name].predict(future_X)
forecast["SARIMA_MW"] = np.asarray(models["SARIMA"].get_forecast(steps=len(future_index)).predicted_mean)
forecast.to_csv(OUTPUT_DIR / "forecast_30_days.csv", index=False)
display(forecast.head())
ax = forecast.set_index("Datetime").plot(figsize=(14, 5), title="30-day hourly forecast by model")
ax.set_ylabel("MW")
plt.show()


## 8. Prepare deployment

Run after all model cells. This saves the fitted model bundle required by the included Streamlit app. The app provides model selection, a 1–30 day horizon, a chart, and a downloadable CSV.


In [ ]:
DEPLOY_DIR.mkdir(exist_ok=True)
joblib.dump({"models": models, "origin": origin, "last_timestamp": hourly.index.max()},
            DEPLOY_DIR / "model_bundle.joblib")
print("Saved deployment/model_bundle.joblib")
print("App:", DEPLOY_DIR / "app.py")
print("Requirements:", DEPLOY_DIR / "requirements.txt")


### Run the app locally

Open a terminal in the deployment folder and run:

    python -m pip install -r requirements.txt
    streamlit run app.py

### Publish the app

Put deployment/app.py, deployment/model_bundle.joblib, and deployment/requirements.txt in a GitHub repository. Create an app from that repository in Streamlit Community Cloud and choose app.py as the entry point. The workbook is not needed at runtime. Publishing requires your account and repository; the notebook prepares the required files but does not publish to an external service.

### Remaining requirement tasks

EDA, four-model evaluation, and a historical 30-day forecast are covered. Deployment artifacts are prepared after running this notebook. The final presentation and review meeting attendance remain separate project tasks.
